In [9]:
from pathlib import Path

def rename_files_custom(folder_path, custom_name, start_number=1):
    """
    Rinomina tutti i file in una cartella a: custom_name(numero).ext

    Args:
        folder_path: percorso cartella
        custom_name: nome base personalizzato
        start_number: numero iniziale (default 1)
    """

    folder = Path(folder_path)

    if not folder.exists():
        print(f"❌ Cartella {folder_path} non trovata")
        return

    # Trova tutti i file
    files = [f for f in folder.iterdir() if f.is_file()]

    print(f"Trovati {len(files)} file da rinominare")
    print(f"Formato: {custom_name}(numero).ext\n")

    # Mostra anteprima
    print("ANTEPRIMA:")
    for idx, file in enumerate(files[:5], start=start_number):
        new_name = f"{custom_name}({idx}){file.suffix}"
        print(f"  {file.name} -> {new_name}")
    if len(files) > 5:
        print(f"  ... e altri {len(files) - 5} file")
    print()

    renamed_count = 0

    for idx, file in enumerate(files, start=start_number):
        ext = file.suffix
        new_name = f"{custom_name}({idx}){ext}"
        new_path = file.parent / new_name

        try:
            file.rename(new_path)
            renamed_count += 1
        except Exception as e:
            print(f"❌ Errore: {e}")

    print(f"\n✓ Rinominati {renamed_count}/{len(files)} file")


# ========== VERSIONE INTERATTIVA ==========

def rename_files_interactive():
    """Versione interattiva che chiede tutto in input"""

    # Chiedi cartella
    folder_path = input("Percorso cartella: ").strip()
    folder = Path(folder_path)

    if not folder.exists():
        print(f"❌ Cartella non trovata!")
        return

    files = [f for f in folder.iterdir() if f.is_file()]
    print(f"\n✓ Trovati {len(files)} file")

    # Chiedi nome base
    custom_name = input("Nome base per i file (es: 'frame', 'img'): ").strip()

    # Chiedi numero iniziale
    start_num = input("Numero iniziale (default 1): ").strip()
    start_num = int(start_num) if start_num else 1

    # Mostra anteprima
    print(f"\nANTEPRIMA (primi 5 file):")
    for idx, file in enumerate(files[:5], start=start_num):
        new_name = f"{custom_name}({idx}){file.suffix}"
        print(f"  {file.name} -> {new_name}")
    if len(files) > 5:
        print(f"  ... e altri {len(files) - 5} file\n")

    # Conferma
    response = input("Procedere? (y/n): ")
    if response.lower() != 'y':
        print("Operazione annullata")
        return

    # Rinomina
    for idx, file in enumerate(files, start=start_num):
        new_name = f"{custom_name}({idx}){file.suffix}"
        new_path = file.parent / new_name
        file.rename(new_path)

    print(f"\n✓ Completato! {len(files)} file rinominati")



In [11]:
import json
from pathlib import Path

# QUESTO è il tuo mapping FINALE che vuoi (dal data.yaml)
correct_map = {
    'furnace': 0, 'archer': 1, 'barbarian': 2, 'battleram': 3, 'bomber': 4, 'bombtower': 5, 'cannon': 6, 'electroSpirit': 7, 'fireSpirit': 8, 'giant': 9, 'goblin': 10, 'goblinCage': 11, 'goblinHut': 12, 'hogRider': 13, 'infernoTower': 14, 'knight': 15, 'miniPekka': 16, 'mortar': 17, 'musketeer': 18, 'skeleton': 19, 'spearGoblin': 20, 'superGoblin': 21, 'tombstone': 22, 'valk': 23, 'wizard': 24, 'allyHP': 25, 'bat': 26, 'enemyHP': 27, 'enemyLevel': 28, 'flyingmachine': 29, 'megaMinion': 30, 'minion': 31, 'skeletonDragon': 32, 'tower': 33
}

label_dir = "../dataset/validation set/labels"

# Label Studio ordina alfabeticamente
labelstudio_order = sorted(correct_map.keys())
print("Ordine alfabetico di Label Studio:")
for idx, name in enumerate(labelstudio_order):
    print(f"  {idx}: {name}")

# Crea mapping: indice_labelstudio -> indice_corretto
index_fix_map = {}
for labelstudio_idx, name in enumerate(labelstudio_order):
    correct_idx = correct_map[name]
    index_fix_map[labelstudio_idx] = correct_idx

print("\n=== MAPPING CORREZIONE ===")
for wrong_idx, correct_idx in sorted(index_fix_map.items()):
    name = labelstudio_order[wrong_idx]
    print(f"  {wrong_idx} ({name}) -> {correct_idx}")

# Correggi tutti i file
label_files = list(Path(label_dir).glob("*.txt"))
print(f"\nCorreggo {len(label_files)} file...")

corrected_count = 0
error_count = 0

for label_file in label_files:
    corrected_lines = []

    try:
        with open(label_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    labelstudio_class_id = int(parts[0])

                    # Correggi l'indice
                    if labelstudio_class_id in index_fix_map:
                        correct_class_id = index_fix_map[labelstudio_class_id]
                        parts[0] = str(correct_class_id)
                        corrected_lines.append(' '.join(parts) + '\n')
                    else:
                        print(f"⚠️  Indice {labelstudio_class_id} non trovato in {label_file.name}")
                        corrected_lines.append(line)

        # Riscrivi file corretto
        with open(label_file, 'w') as f:
            f.writelines(corrected_lines)

        corrected_count += 1

    except Exception as e:
        print(f"❌ Errore processando {label_file.name}: {e}")
        error_count += 1

print(f"\n✓ Corretto {corrected_count}/{len(label_files)} file!")
if error_count > 0:
    print(f"❌ {error_count} errori")

Ordine alfabetico di Label Studio:
  0: allyHP
  1: archer
  2: barbarian
  3: bat
  4: battleram
  5: bomber
  6: bombtower
  7: cannon
  8: electroSpirit
  9: enemyHP
  10: enemyLevel
  11: fireSpirit
  12: flyingmachine
  13: furnace
  14: giant
  15: goblin
  16: goblinCage
  17: goblinHut
  18: hogRider
  19: infernoTower
  20: knight
  21: megaMinion
  22: miniPekka
  23: minion
  24: mortar
  25: musketeer
  26: skeleton
  27: skeletonDragon
  28: spearGoblin
  29: superGoblin
  30: tombstone
  31: tower
  32: valk
  33: wizard

=== MAPPING CORREZIONE ===
  0 (allyHP) -> 25
  1 (archer) -> 1
  2 (barbarian) -> 2
  3 (bat) -> 26
  4 (battleram) -> 3
  5 (bomber) -> 4
  6 (bombtower) -> 5
  7 (cannon) -> 6
  8 (electroSpirit) -> 7
  9 (enemyHP) -> 27
  10 (enemyLevel) -> 28
  11 (fireSpirit) -> 8
  12 (flyingmachine) -> 29
  13 (furnace) -> 0
  14 (giant) -> 9
  15 (goblin) -> 10
  16 (goblinCage) -> 11
  17 (goblinHut) -> 12
  18 (hogRider) -> 13
  19 (infernoTower) -> 14
  20 (k

In [10]:
rename_files_custom(
    folder_path="all_segments/fire-spirit",
    custom_name="_fireSpirit",  # <-- NOME PERSONALIZZATO
    start_number=0
)

Trovati 29 file da rinominare
Formato: _fireSpirit(numero).ext

ANTEPRIMA:
  _fireSpirit(0).png -> _fireSpirit(0).png
  _fireSpirit(1).png -> _fireSpirit(1).png
  _fireSpirit(10).png -> _fireSpirit(2).png
  _fireSpirit(11).png -> _fireSpirit(3).png
  _fireSpirit(12).png -> _fireSpirit(4).png
  ... e altri 24 file

❌ Errore: [WinError 183] Cannot create a file when that file already exists: 'all_segments\\fire-spirit\\_fireSpirit(10).png' -> 'all_segments\\fire-spirit\\_fireSpirit(2).png'
❌ Errore: [WinError 183] Cannot create a file when that file already exists: 'all_segments\\fire-spirit\\_fireSpirit(11).png' -> 'all_segments\\fire-spirit\\_fireSpirit(3).png'
❌ Errore: [WinError 183] Cannot create a file when that file already exists: 'all_segments\\fire-spirit\\_fireSpirit(12).png' -> 'all_segments\\fire-spirit\\_fireSpirit(4).png'
❌ Errore: [WinError 183] Cannot create a file when that file already exists: 'all_segments\\fire-spirit\\_fireSpirit(13).png' -> 'all_segments\\fire-spir